# STAC Query Examples
## Demonstrating the value of STAC for spatiotemporal data discovery

### Setup: Install required libraries

In [ ]:
import requests
import json
import pandas as pd
from pprint import pprint

# STAC server URL
STAC_URL = "http://89.47.191.90:3000"

print(f"Connecting to STAC server at: {STAC_URL}")

### Example 1: List all collections

In [ ]:
# Get all collections
response = requests.get(f"{STAC_URL}/collections")
collections = response.json()

print(f"Found {len(collections['collections'])} collection(s)\n")

for collection in collections['collections']:
    print(f"Collection ID: {collection['id']}")
    print(f"Title: {collection['title']}")
    print(f"Description: {collection['description']}")
    print(f"License: {collection['license']}")
    print()

### Example 2: Search by spatial extent (bbox)

In [ ]:
# Search for items within a bounding box
bbox = [172.91, 1.34, 172.96, 1.37]  # [minx, miny, maxx, maxy]

search_query = {
    "collections": ["simple-collection"],
    "bbox": bbox
}

response = requests.post(f"{STAC_URL}/search", json=search_query)
results = response.json()

print(f"Found {results['numberMatched']} item(s) in bbox: {bbox}\n")

for feature in results['features']:
    print(f"Item ID: {feature['id']}")
    print(f"Geometry: {feature['geometry']['type']}")
    print(f"Properties: {json.dumps(feature['properties'], indent=2)}")
    print()

### Example 3: Search by date range

In [ ]:
# Search for items within a date range
search_query = {
    "collections": ["simple-collection"],
    "datetime": "2020-12-01T00:00:00Z/2020-12-31T23:59:59Z"
}

response = requests.post(f"{STAC_URL}/search", json=search_query)
results = response.json()

print(f"Found {results['numberMatched']} item(s) in December 2020\n")

for feature in results['features']:
    datetime_str = feature['properties'].get('datetime', feature['properties'].get('start_datetime'))
    print(f"Item ID: {feature['id']}")
    print(f"DateTime: {datetime_str}")
    print()

### Example 4: Combined spatial + temporal search

In [ ]:
# Search with both bbox and datetime
search_query = {
    "collections": ["simple-collection"],
    "bbox": [172.91, 1.34, 172.96, 1.37],
    "datetime": "2020-12-01T00:00:00Z/2020-12-31T23:59:59Z"
}

response = requests.post(f"{STAC_URL}/search", json=search_query)
results = response.json()

print(f"Found {results['numberMatched']} item(s) matching both spatial and temporal criteria\n")

for feature in results['features']:
    props = feature['properties']
    print(f"Item: {feature['id']}")
    print(f"  Platform: {props.get('platform')}")
    print(f"  GSD: {props.get('gsd')} m")
    print(f"  Start DateTime: {props.get('start_datetime')}")
    print()

### Example 5: Access the actual data assets

In [ ]:
# Search and get asset URLs
search_query = {
    "collections": ["simple-collection"],
    "bbox": [172.91, 1.34, 172.96, 1.37]
}

response = requests.post(f"{STAC_URL}/search", json=search_query)
results = response.json()

print("Available Data Assets:\n")

for feature in results['features']:
    print(f"Item: {feature['id']}")
    assets = feature.get('assets', {})
    
    for asset_name, asset_info in assets.items():
        print(f"  - {asset_name}:")
        print(f"    URL: {asset_info['href']}")
        print(f"    Type: {asset_info.get('type')}")
        print(f"    Title: {asset_info.get('title')}")
    print()

### Example 6: Query with filters (e.g., cloud cover)

In [ ]:
# Search with query filters
search_query = {
    "collections": ["simple-collection"],
    "query": {
        "gsd": {"lte": 1.0}  # Find items with GSD <= 1.0 meter
    }
}

response = requests.post(f"{STAC_URL}/search", json=search_query)
results = response.json()

print(f"Found {results['numberMatched']} item(s) with GSD <= 1.0 m\n")

for feature in results['features']:
    props = feature['properties']
    print(f"Item: {feature['id']}")
    print(f"  GSD: {props.get('gsd')} m")
    print(f"  Platform: {props.get('platform')}")
    print()

### Example 7: Create a DataFrame for analysis

In [ ]:
# Get all items and create a DataFrame
search_query = {
    "collections": ["simple-collection"]
}

response = requests.post(f"{STAC_URL}/search", json=search_query)
results = response.json()

# Extract data for DataFrame
items_data = []

for feature in results['features']:
    props = feature['properties']
    items_data.append({
        'item_id': feature['id'],
        'platform': props.get('platform'),
        'gsd': props.get('gsd'),
        'start_datetime': props.get('start_datetime'),
        'end_datetime': props.get('end_datetime'),
        'num_assets': len(feature.get('assets', {}))
    })

df = pd.DataFrame(items_data)
print("Items Summary:")
print(df)
print()
print(f"Total items: {len(df)}")
print(f"Unique platforms: {df['platform'].nunique()}")

### Example 8: Download a thumbnail (bonus)

In [ ]:
# Get an item and download its thumbnail
search_query = {
    "collections": ["simple-collection"],
    "limit": 1
}

response = requests.post(f"{STAC_URL}/search", json=search_query)
results = response.json()

if results['features']:
    feature = results['features'][0]
    assets = feature.get('assets', {})
    
    if 'thumbnail' in assets:
        thumbnail_url = assets['thumbnail']['href']
        print(f"Downloading thumbnail from: {thumbnail_url}")
        
        img_response = requests.get(thumbnail_url)
        with open('thumbnail.jpg', 'wb') as f:
            f.write(img_response.content)
        
        print("✅ Thumbnail saved as thumbnail.jpg")
        print(f"File size: {len(img_response.content)} bytes")

## Why STAC Matters

✅ **Standardized queries** across different data providers  
✅ **Spatial + temporal search** built-in  
✅ **Direct links to data** (no separate file server lookup)  
✅ **Self-documenting** (all metadata in response)  
✅ **Interoperable** (works with any STAC-compliant server)  
✅ **Easy integration** with Python/pandas for analysis